# Prueba 2 — Calibración de WPE (taps × delay × RT60)

Caracteriza el hiperparámetro de WPE barriendo `wpe_taps` [3,5,7,10] × `wpe_delay`
[1,2,3] × RT60 {160,360,610} ms. De una sola corrida se obtienen **dos cosas**:

1. **`delay*`**: el mejor `delay` en la fila `taps=5` (fijo por memoria del FPGA).
   Es el valor que se copia a las Pruebas 3 y 4.
2. **Honestidad de taps**: cuánto se gana con `taps=10` (techo no restringido) vs
   `taps=5` (HW), y si eso depende del RT. Documenta la restricción de HW con
   transparencia. (Absorbe el panel de taps; no hace falta un notebook aparte.)

`t_early` está **fijo en 8 ms**, desacoplado de `wpe_delay`, así que todas las
combinaciones taps×delay se evalúan contra la MISMA referencia early (comparables).

**Cómo ejecutar:** *Setup* una vez por sesión, luego *Ejecución* y *Selección*.

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')


In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

In [ ]:
import os

WHL = "/content/drive/MyDrive/colab_wheels"   # cache persistente de wheels en Drive

# CONSTRUIR (git+ para las libs de GitHub).
BUILD = [
    "noisereduce", "mir_eval", "pystoi", "pesq", "paderbox", "ai_edge_litert",
    "git+https://github.com/fgnt/pb_bss.git",
    "git+https://github.com/LCAV/pyroomacoustics.git",
    "git+https://github.com/fgnt/nara_wpe.git",
    "git+https://github.com/fakufaku/fast_bss_eval.git",
]
# INSTALAR desde cache: NOMBRES (no git+, si no pip vuelve a clonar).
INSTALL = [
    "noisereduce", "mir_eval", "pystoi", "pesq", "paderbox", "ai_edge_litert",
    "pb_bss", "pyroomacoustics", "nara_wpe", "fast_bss_eval",
]

# Reconstruye el cache SOLO si la lista de paquetes cambio (manifest) -> se
# autocura si agrego/saco un paquete, sin tener que borrar el cache a mano.
manifest = os.path.join(WHL, ".manifest.txt")
key = "\n".join(sorted(BUILD))
need_build = (not os.path.isfile(manifest)) or open(manifest).read() != key

if need_build:
    os.makedirs(WHL, exist_ok=True)
    print("[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...")
    !pip wheel --wheel-dir=$WHL {" ".join(BUILD)}
    with open(manifest, "w") as fh:
        fh.write(key)
    print("[*] Cache actualizado en", WHL)

!pip install --no-index --find-links=$WHL {" ".join(INSTALL)}
print("[*] Paquetes instalados desde el cache de Drive.")
# Si Colab actualiza Python y falla un import:  !rm -rf $WHL  (se reconstruye solo)

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

## Ejecución del sweep

In [ ]:
import sys
import os
import numpy as np
import shutil
from datetime import datetime


repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')
for p in (repo_root, src_path):
    if p not in sys.path: sys.path.append(p)
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    print("[!] TensorFlow no detectado. DTLN-mono desactivado.")
    TFLITE_AVAILABLE = False

from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import NM_MVDR
from propagation.mird_loader import MirdDatasetProvider

model_1_path = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
model_2_path = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")
interpreter_1 = interpreter_2 = None
if TFLITE_AVAILABLE and os.path.exists(model_1_path) and os.path.exists(model_2_path):
    interpreter_1 = tf.lite.Interpreter(model_path=model_1_path); interpreter_1.allocate_tensors()
    interpreter_2 = tf.lite.Interpreter(model_path=model_2_path); interpreter_2.allocate_tensors()
    print("[*] DTLN TFLite OK.")
else:
    print("[*] Sin DTLN-mono (NM-MVDR igual usa su mascara interna).")

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

# ===================== PERILLAS =====================
DURATION = 15   # lever de tiempo (bajalo si no entra la sesion)
# ===================================================

base_config = {
    'fs': 16000, 'duration': DURATION, 't_early': 0.008,   # t_early FIJO (8 ms)
    'array_center': [3.0, 3.0, 1.2], 'mird_spacing': "3-3-3-8-3-3-3",
    'snr_db': 60.0,
    'source_path': os.path.join(input_dir, "p002_emo_adoration_sentences.wav"),
    'interf_paths': [
        os.path.join(input_dir, "hairdryer_07_SH_MKH800.wav"),
        os.path.join(input_dir, "flute_music.wav"),
    ],
    'wpe_taps': 5, 'wpe_delay': 1, 'wpe_alpha': 0.9999,   # fallbacks (los pisa el grid)
    'wpe_stft_size': 512, 'wpe_stft_shift': 128,
    'stft_window': 512, 'stft_overlap': 384,
    'dtln_model_path': model_1_path,
    'eval_references': ['early'],
}

# Barrido taps x delay x RT (escena de estres: 1-2 interferentes, iSIR=0).
param_grid = {
    'rt60':          [0.160, 0.360, 0.610],   # <-- barrido de RT (estabilidad de delay*)
    'target_angle':  [0],
    'target_dist':   [1.0],
    'interf_configs':[
        [(45, 1.0)],
        [(-90, 1.0)],
        [(45, 1.0), (-90, 1.0)],
    ],
    'isir_db':       [0],
    'mismatch_gain': [0], 'mismatch_phase': [0],
    'use_wpe':       [True],
    'wpe_taps':      [3, 5, 7, 10],   # <-- taps=5 (HW) + techo taps=10
    'wpe_delay':     [1, 2, 3],       # <-- delay a calibrar
    'error_angle_deg':[0.0], 'error_distance_m':[0.0],
}

processors_dict = {
    "NM-MVDR": NM_MVDR(min_loading=1e-6, alpha=0.99),
}

n_cells = 3*3*4*3   # rt60 x interf x taps x delay
print("="*60)
print(f"CALIBRACION WPE (taps x delay x RT) | celdas={n_cells}")
print("="*60)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")
temp_output_dir  = f"/content/results_temp/P2_calibracion_wpe_{RUN_TAG}"
drive_output_dir = f"/content/drive/MyDrive/Tesis_Beamformers/results/P2_calibracion_wpe_{RUN_TAG}"
os.makedirs(temp_output_dir, exist_ok=True); os.makedirs(drive_output_dir, exist_ok=True)

df_E0 = run_mird_grid_search(
    grid_params=param_grid, dataset_provider=provider, processors=processors_dict,
    scene_base_config=base_config, output_dir=temp_output_dir,
    interpreter_1=interpreter_1, interpreter_2=interpreter_2, save_catalog=False,
)

print("\n[INFO] Sincronizando a Drive...")
shutil.copytree(temp_output_dir, drive_output_dir, dirs_exist_ok=True)
print(f"\n[EXITO] Prueba 2 (calibracion WPE) guardada en {drive_output_dir}")

## Selección del óptimo

In [ ]:
# --- Seleccion: delay* (fila taps=5) + honestidad de taps + estabilidad con RT ---
import pandas as pd
import numpy as np

df = pd.read_csv(os.path.join(drive_output_dir, "mird_benchmark_metrics.csv"))
df = df[df["processor"] == "NM-MVDR"].copy()

def pivot(metric):
    return df.groupby(["wpe_taps", "wpe_delay"])[metric].mean().unstack("wpe_delay")

# 1) SUPERFICIE taps x delay (media sobre RT + escenas) para metricas clave.
print("SUPERFICIE taps(filas) x delay(cols)  |  media sobre RT + escenas")
for m, lbl, d in [("Delta_tot_PESQ_early","PESQ(early)","mayor mejor"),
                  ("Delta_tot_SDR_early","SDR","mayor mejor"),
                  ("Delta_tot_CD_early","CD","MENOR mejor")]:
    if m in df.columns:
        print(f"\n--- {lbl}  ({d}) ---")
        print(pivot(m).round(3).to_string())

# 2) delay*: fila taps=5, todas las metricas por delay.
m5 = ["Delta_tot_PESQ_early","Delta_tot_STOI_early","Delta_tot_SDR_early",
      "Delta_tot_SIR_early","Delta_tot_CD_early"]
m5 = [c for c in m5 if c in df.columns]
print("\n" + "="*60)
print("2) taps=5 | metricas por delay  ->  ELEGIR delay* aca")
print("="*60)
print(df[df.wpe_taps==5].groupby("wpe_delay")[m5].mean().round(3).to_string())

# 3) HONESTIDAD de taps: a delay fijo, taps=5 vs taps=10 (fijar DELAY_SHOW = delay*).
DELAY_SHOW = 2   # <-- poner delay* elegido en el bloque 2
mt = ["Delta_tot_PESQ_early","Delta_tot_SDR_early","Delta_tot_CD_early"]
mt += [c for c in ["Delta_wpe_PESQ_early","Delta_wpe_SDR_early"] if c in df.columns]
mt = [c for c in mt if c in df.columns]
print("\n" + "="*60)
print(f"3) delay={DELAY_SHOW} | sensibilidad a taps (honestidad HW: taps=5 vs 10)")
print("="*60)
print(df[df.wpe_delay==DELAY_SHOW].groupby("wpe_taps")[mt].mean().round(3).to_string())

# 4) ESTABILIDAD de delay* con RT: taps=5, PESQ por (RT x delay).
if "Delta_tot_PESQ_early" in df.columns:
    print("\n" + "="*60)
    print("4) taps=5 | PESQ(early) por RT(filas) x delay(cols)  ->  delay* estable con RT?")
    print("="*60)
    print(df[df.wpe_taps==5].groupby(["rt60","wpe_delay"])["Delta_tot_PESQ_early"]
            .mean().unstack("wpe_delay").round(3).to_string())

print("\n>>> delay*: bloque 2 (fila taps=5), mirando TODAS las metricas + estabilidad con RT (bloque 4).")
print(">>> Honestidad HW: bloque 3 documenta lo que se pierde por fijar taps=5 vs taps=10.")
print(">>> Copia delay* a WPE_DELAY de las Pruebas 3 y 4.")